In [ ]:
!pip install numpy ipympl librosa scipy

In [ ]:
import numpy as np
import librosa as lr
from numpy import pi
ε = 1e-10
from scipy import signal
import random
import matplotlib.pyplot as plt
%matplotlib inline

# Вариант 6

### Задание

1. Ознакомиться с теоретической частью.

1. Для тестовых музыкальных файлов реализовать мел-спектраграмму:

    - [x] реализовать алгоритм вычисления мел-спектраграммы
  
    - [x] реализовать мел-спектраграмму с помощью готовых библиотек
  
    - объяснить полученный результат
  
1. Для тестовых музыкальных файлов вычислить спектральные признаки аудиосигнала:

    - самостоятельно:
  
        - [ ] частота пересечения нуля
      
        - [x] спектральная ширина
     
    - с помощью библиотек:
  
        - [ ] любой из методы на выбор
     
        - [ ] любой из методы на выбор
     
        - [ ] любой из методы на выбор

    - объяснить полученный результат
  
1. Написать функцию, которая будет смешивать чистый голос и шум по `SNR = 0,3..15 дБ`.

1. Сравнить методы оценки качества звука:

    - самостоятельно:
  
        - [ ] SNR
     
        - [ ] SDR
     
    - с помощью библиотек:
  
        - [ ] SI-SDR
     
        - [ ] PESQ
     
        - [ ] NISQA
     
        - [ ] DNSMOS
     
    - [ ] вывести в виде таблицы:
  
      тестовый файл | объективные оценки (SNR, SDR, etc.) | субъективная оценка
      ---           |---                                  |---
  
1. Прогнать через шумоподавление:

    - [ ] установить модель шумоподавления (DeepFilterNet2 или более актуальную)
  
    - [ ] повторить тесты из п.5
      

In [ ]:
samples, sr = lr.load('africa-toto.wav', sr=None)

### МЕЛ-Спектрограмма

In [ ]:
def plot_lr_mel_spectr(samples, mel_count=128):
    mel_spectr = lr.feature.melspectrogram(y=samples, sr=sr, n_mels=mel_count)
    mel_spectr_db = lr.power_to_db(mel_spectr, top_db=None)
    
    plt.figure(figsize=(12, 4))
    lr.display.specshow(mel_spectr_db, sr=sr, x_axis='time', y_axis='mel', cmap="magma", )
    plt.colorbar(format="%+2.0f dB")
    plt.tight_layout()

plot_lr_mel_spectr(samples)
plt.show()

In [ ]:
def plot_our_mel_spectr(samples, n_mels=128):
    def mel(hz): return 2595 * np.log10(1 + hz/700)
    def unmel(mel): return 700 * (10**(mel/2595) - 1)
    def db(x): return 10 * np.log10(x + ε) # fix bug when `x` can be `-0.0`

    def mel_filters(count: int, freqs):
        f_min_mel = mel(min(freqs))
        f_max_mel = mel(max(freqs))
        f_mel_step = (f_max_mel - f_min_mel) / (count+1)

        filters, mel_freqs = [], []
        for f_mel in np.arange(f_min_mel + f_mel_step, f_max_mel, f_mel_step):
            f_mid, f_start, f_end = unmel(f_mel), unmel(f_mel-f_mel_step), unmel(f_mel+f_mel_step)
            mid_sample, start_sample, end_sample = np.abs(freqs-f_mid).argmin(), np.abs(freqs-f_start).argmin(), np.abs(freqs-f_end).argmin()

            tri_leftwidth, tri_rightwidth = max(1, mid_sample-start_sample), max(1, end_sample-mid_sample)
            tri = np.concatenate((
                np.linspace(0, 1, num=tri_leftwidth),
                np.linspace(1, 0, num=tri_rightwidth)
            ))

            filter = np.pad(tri, pad_width=(start_sample, len(freqs)-start_sample-tri_leftwidth-tri_rightwidth), constant_values=0)
            filters.append(filter)
            mel_freqs.append(f_mel)

        return np.array(filters), np.array(mel_freqs)
        
    # https://www.youtube.com/watch?v=-Yxj3yfvY-4
    freqs, times, stft = signal.stft(samples, sr, nperseg=1024) # TODO? probably implement ourselves
    ampls = np.abs(stft)**2
    
    filters, mel_freqs = mel_filters(n_mels, freqs)
    mel_ampls = np.dot(filters, ampls)

    mel_ampls = db(mel_ampls)

    plt.figure(figsize=(12, 4))
    plt.pcolormesh(times, mel_freqs, mel_ampls, cmap="magma")
    plt.ylabel('Mel')
    plt.xlabel('Time')
    plt.colorbar(format="%+2.0f dB")
    plt.tight_layout()

plot_our_mel_spectr(samples)
plt.show()

### Спектральные признаки

In [ ]:
def spectral_centroid(freqs, times, ampls):
    M = len(freqs)
    return [
        sum(ampls[k][l] * freqs[k] for k in range(M))
         / (sum(ampls[k][l] for k in range(M)) + ε)
    for l in range(len(times))]

def plot_spectral_width(samples):
    def spectral_width(freqs, times, ampls):
        centroid = spectral_centroid(freqs, times, ampls)
        
        M = len(freqs)
        return [
            np.sqrt(
                sum(ampls[k][l] * (freqs[k]-centroid[l])**2 for k in range(M))
                 / (sum(ampls[k][l] for k in range(M)) + ε)
            )
        for l in range(len(times))]

    freqs, times, stft = signal.stft(samples, sr, nperseg=1024)
    ampls = np.abs(stft)
    sw = spectral_width(freqs, times, ampls)
    
    plot_lr_mel_spectr(samples)
    plt.plot(times, sw, color="yellow")

plot_spectral_width(samples)
plt.show()